# Truncated Policy Iteration — FrozenLake

**MSSV:** `<MaSo>` &nbsp;&nbsp; **Họ tên:** `<HoTen>`

## Ý tưởng
Policy Iteration gốc: **Evaluate** (lặp đến khi V hội tụ) → **Improve** → lặp lại.
**Truncated** Policy Iteration: bước Evaluate chỉ chạy **k lần quét (sweep)** rồi Improve ngay → nhanh hơn, vẫn hội tụ về policy tối ưu.

```
policy π₀ (ngẫu nhiên), V = 0
loop:
    V ← chạy k sweep Bellman expectation với π      (truncated evaluation)
    π' ← greedy theo V                              (policy improvement)
    nếu π' == π và V gần như không đổi → dừng
```

| | Policy Iteration | **Truncated PI** | Value Iteration |
|---|---|---|---|
| Số sweep evaluate mỗi vòng | đến khi hội tụ | **k** | 1 |


In [16]:
# !pip install gymnasium   # bỏ comment nếu chưa cài
import numpy as np
import gymnasium as gym

## 1. Môi trường FrozenLake
`S`: start, `F`: băng, `H`: hố (thua), `G`: đích (+1). Action: 0=Left, 1=Down, 2=Right, 3=Up.

In [17]:
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
P = env.unwrapped.P          # P[s][a] = [(prob, next_state, reward, done), ...]
N_STATES = env.observation_space.n
N_ACTIONS = env.action_space.n
GAMMA = 0.99

print("Số state:", N_STATES, "| Số action:", N_ACTIONS)
print("P[14][2] (đứng cạnh G, đi phải):", P[14][2])

Số state: 16 | Số action: 4
P[14][2] (đứng cạnh G, đi phải): [(0.33333333333333337, 14, 0, False), (0.3333333333333333, 15, 1, True), (0.33333333333333337, 10, 0, False)]


## 2. Hàm hiển thị (có sẵn)

In [18]:
ARROWS = ["←", "↓", "→", "↑"]
DESC = env.unwrapped.desc.astype(str)

def show_values(V):
    print(np.round(V.reshape(DESC.shape), 3))

def show_policy(policy):
    grid = []
    for s in range(N_STATES):
        cell = DESC.flat[s]
        grid.append(cell if cell in "HG" else ARROWS[policy[s]])
    print(np.array(grid).reshape(DESC.shape))

## 3. Tính Q(s, a) từ V (helper dùng chung)
$Q(s,a) = \sum_{s'} p(s'|s,a)\,[\,r + \gamma V(s')\,]$ — nếu `done` thì không cộng $\gamma V(s')$.

In [19]:
def q_from_v(V, s, gamma=GAMMA):
    q = np.zeros(N_ACTIONS)
    for a in range(N_ACTIONS):
        for prob, s_next, reward, done in P[s][a]:
            q[a] += prob * (reward + gamma * V[s_next] * (not done))
    return q

## 4. ✍️ TODO 1 — Truncated Policy Evaluation
Chạy đúng **k sweep**. Mỗi sweep, với mọi state: `V[s] = Q(s, policy[s])` (policy deterministic).

Gợi ý: dùng `q_from_v(V, s)[policy[s]]`. Cập nhật tại chỗ (in-place) cũng được.

In [20]:
def truncated_policy_evaluation(policy, V, k, gamma=GAMMA):
    V = V.copy()
    # TODO: lặp k lần, mỗi lần quét qua tất cả state và cập nhật V[s]
    for k in range(k):
        for s in range(N_STATES):
            V[s] = q_from_v(V, s)[policy[s]]
    return V

## 5. ✍️ TODO 2 — Policy Improvement
Với mọi state: `policy[s] = argmax_a Q(s, a)`.

In [21]:
def policy_improvement(V, gamma=GAMMA):
    policy = np.zeros(N_STATES, dtype=int)
    # TODO: với mỗi state s, chọn action có Q lớn nhất
    policy = np.array([np.argmax(q_from_v(V, s)) for s in range(N_STATES)])
    return policy

## 6. ✍️ TODO 3 — Vòng lặp Truncated Policy Iteration
Dừng khi **cả hai**: policy mới == policy cũ **và** `max|V_new - V| < theta`. Trả về `policy, V, số vòng`.

⚠️ Bẫy: chỉ check `ΔV < theta` sẽ dừng ngay vòng 1, vì policy ban đầu (toàn ←) không bao giờ tới G nên V = 0 → ΔV = 0.
Gợi ý: `np.array_equal(new_policy, policy)`.

In [22]:
def truncated_policy_iteration(k=5, theta=1e-8, gamma=GAMMA, max_iters=10_000):
    V = np.zeros(N_STATES)
    policy = np.zeros(N_STATES, dtype=int)

    for i in range(1, max_iters + 1):
        # TODO 3a: evaluate k sweep -> V_new
        # TODO 3b: improve -> new_policy
        # TODO 3c: nếu policy ổn định VÀ ΔV < theta -> return new_policy, V_new, i
        # TODO 3d: cập nhật policy = new_policy, V = V_new
        V_new = truncated_policy_evaluation(policy, V, k, gamma)
        new_policy = policy_improvement(V_new, gamma)
        if np.array_equal(new_policy, policy) and np.max(np.abs(V_new - V)) < theta:
            return new_policy, V_new, i
        policy = new_policy
        V = V_new

    return policy, V, max_iters

## 7. Chạy & kiểm tra

In [23]:
policy, V, n_iters = truncated_policy_iteration(k=5)
print(f"Hội tụ sau {n_iters} vòng\n")
print("Value function:"); show_values(V)
print("\nPolicy:"); show_policy(policy)

Hội tụ sau 75 vòng

Value function:
[[0.542 0.499 0.471 0.457]
 [0.558 0.    0.358 0.   ]
 [0.592 0.643 0.615 0.   ]
 [0.    0.742 0.863 0.   ]]

Policy:
[['←' '↑' '↑' '↑']
 ['←' 'H' '←' 'H']
 ['↑' '↓' '←' 'H']
 ['H' '→' '↓' 'G']]


In [24]:
def evaluate_win_rate(policy, n_episodes=1000, seed=42):
    wins = 0
    for ep in range(n_episodes):
        s, _ = env.reset(seed=seed + ep)
        done = False
        while not done:
            s, r, terminated, truncated, _ = env.step(policy[s])
            done = terminated or truncated
        wins += r
    return wins / n_episodes

print("Win rate (policy học được):", evaluate_win_rate(policy))
print("Win rate (policy ngẫu nhiên):", evaluate_win_rate(np.random.randint(0, 4, N_STATES)))

Win rate (policy học được): 0.752
Win rate (policy ngẫu nhiên): 0.0


## 8. So sánh ảnh hưởng của k
`k=1` ≈ Value Iteration, `k` lớn ≈ Policy Iteration.

In [25]:
import time
for k in [1, 3, 5, 10, 50]:
    t = time.perf_counter()
    pol, _, it = truncated_policy_iteration(k=k)
    dt = time.perf_counter() - t
    print(f"k={k:>3} | vòng lặp: {it:>5} | tổng sweep: {it*k:>6} | thời gian: {dt:.3f}s | win rate: {evaluate_win_rate(pol):.3f}")

k=  1 | vòng lặp:   325 | tổng sweep:    325 | thời gian: 0.115s | win rate: 0.752
k=  3 | vòng lặp:   118 | tổng sweep:    354 | thời gian: 0.054s | win rate: 0.752
k=  5 | vòng lặp:    75 | tổng sweep:    375 | thời gian: 0.061s | win rate: 0.752
k= 10 | vòng lặp:    41 | tổng sweep:    410 | thời gian: 0.050s | win rate: 0.752
k= 50 | vòng lặp:    14 | tổng sweep:    700 | thời gian: 0.077s | win rate: 0.752


In [26]:
np.save("policy.npy", policy)

## 9. Nhận xét
_(Tự viết 3–5 dòng: k ảnh hưởng thế nào tới số vòng/thời gian? Policy cuối có giống nhau không? Vì sao win rate không đạt 100%?)_